In [ ]:
!pip install "numpy<2"
!pip install scikit-surprise
!pip install sentence-transformers
# KHAI BÁO THƯ VIỆN
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import train_test_split
from sentence_transformers import SentenceTransformer # Thêm thư viện Deep Learning
from google.colab import files
import math

In [ ]:
# ==========================================
# 1: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
print("1. Đang tải và dọn dẹp dữ liệu...")

# Xử lý Anime
anime_df = pd.read_csv('/content/drive/MyDrive/Hybrid Anime Recommender System/anime-filtered.csv')
anime_df = anime_df.rename(columns={'Name': 'name', 'Genres': 'genre'})

# Điền giá trị rỗng cho các cột Text
anime_df['genre'] = anime_df['genre'].fillna('')
anime_df['sypnopsis'] = anime_df['sypnopsis'].fillna('')

# Xử lý Rating (Dùng 2 triệu dòng, ép kiểu để tối ưu RAM)
rating_df = pd.read_csv('/content/drive/MyDrive/Hybrid Anime Recommender System/user-filtered.csv',
                        nrows=2000000,
                        dtype={'user_id': 'int32', 'anime_id': 'int32', 'rating': 'int8'})

# Xóa điểm 0 (Người dùng lưu phim nhưng lười chấm điểm)
rating_df = rating_df[rating_df['rating'] > 0]

# Lọc dữ liệu thưa (Chỉ lấy User > 50 phim và Anime > 50 lượt vote)
min_anime_ratings = 50
filter_animes = rating_df['anime_id'].value_counts() > min_anime_ratings
filter_animes = filter_animes[filter_animes].index.tolist() # mang gom nhung id user > 50 phim

min_user_ratings = 50
filter_users = rating_df['user_id'].value_counts() > min_user_ratings
filter_users = filter_users[filter_users].index.tolist()

rating_df_clean = rating_df[(rating_df['anime_id'].isin(filter_animes)) & (rating_df['user_id'].isin(filter_users))]
print(f"-> Tổng số rating tinh khiết để train AI: {len(rating_df_clean)}")

1. Đang tải và dọn dẹp dữ liệu...
-> Tổng số rating tinh khiết để train AI: 1019961


In [ ]:
# # co the bo sung train theo mo ta, dao dien, ...
# #2: HUẤN LUYỆN CONTENT-BASED
# print("\n2. Đang xây dựng ma trận nội dung (TF-IDF & Cosine)...")

# # ÉP MA TRẬN NHỎ LẠI
# valid_anime_ids = rating_df_clean['anime_id'].unique()
# anime_df = anime_df[anime_df['anime_id'].isin(valid_anime_ids)]

# tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
# tfidf_matrix = tf.fit_transform(anime_df['genre']) # Một ma trận toán học khổng lồ toàn số thập phân. Mỗi hàng là 1 anime, mỗi cột là 1 từ khoá, giá trị là điểm số TF-IDF.
# cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix) # đưa vào 2 lần để nó tự lấy từng anime so sánh với toàn bộ các anime còn lại

# # Tạo từ điển tra cứu nhanh
# anime_df = anime_df.reset_index(drop=True) # (VD: 0, 1, 4, 5, 9...), đánh số thứ tự hàng lại từ đầu cho đẹp: 0, 1, 2, 3... drop=True nghĩa là vứt bỏ luôn cái cột số thứ tự cũ đi.
# indices = pd.Series(anime_df.index, index=anime_df['name'])        # Tra theo Tên
# id_to_index = pd.Series(anime_df.index, index=anime_df['anime_id']) # Tra theo ID

In [ ]:
# ==========================================
# 2: HUẤN LUYỆN CONTENT-BASED (CẢ 2 MÔ HÌNH)
# ==========================================
print("\n2. Đang xây dựng hệ thống đặc trưng (Text & Heuristics)...")

valid_anime_ids = rating_df_clean['anime_id'].unique()
anime_df = anime_df[anime_df['anime_id'].isin(valid_anime_ids)].copy()

# Tạo Nồi lẩu văn bản
anime_df['metadata_soup'] = anime_df['genre'] + " " + anime_df['genre'] + " " + anime_df['sypnopsis']

# --- MÔ HÌNH 1: CỔ ĐIỂN (TF-IDF) ---
print("-> Đang huấn luyện TF-IDF (Classic)...")
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, stop_words='english')
tfidf_matrix = tf.fit_transform(anime_df['metadata_soup'])
tfidf_cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# --- MÔ HÌNH 2: DEEP LEARNING (SENTENCE-BERT) ---
print("-> Đang huấn luyện Deep Learning (Sentence-BERT)... Có thể mất 2-3 phút.")
# Dùng model 'all-MiniLM-L6-v2' (Nhỏ, nhẹ, chạy cực nhanh nhưng rất thông minh)
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
# Biến 10.000 đoạn tóm tắt thành Ma trận Vector đa chiều
bert_embeddings = bert_model.encode(anime_df['metadata_soup'].tolist(), show_progress_bar=True)
# Tính độ tương đồng Cosine trên Vector của BERT
bert_cosine_sim = cosine_similarity(bert_embeddings, bert_embeddings)

# --- XỬ LÝ CHỈ SỐ STATS ---
anime_df['Score'] = anime_df['Score'].fillna(0)
anime_df['Dropped'] = anime_df['Dropped'].fillna(0)
anime_df['Members'] = anime_df['Members'].replace(0, 1)
anime_df['drop_rate'] = anime_df['Dropped'] / anime_df['Members']
scaler = MinMaxScaler()
anime_df['normalized_score'] = scaler.fit_transform(anime_df[['Score']])
anime_df['quality_weight'] = 1.0 + (anime_df['normalized_score'] * 0.2) - (anime_df['drop_rate'] * 1.5) # quality_weight để thưởng điểm cho phim Rate cao và trừ điểm phim Drop nhiều.
anime_df['quality_weight'] = anime_df['quality_weight'].clip(lower=0.5)

# --- TẠO TỪ ĐIỂN TRA CỨU ---
anime_df = anime_df.reset_index(drop=True)
id_to_index = pd.Series(anime_df.index, index=anime_df['anime_id'])


2. Đang xây dựng hệ thống đặc trưng (Text & Heuristics)...
-> Đang huấn luyện TF-IDF (Classic)...
-> Đang huấn luyện Deep Learning (Sentence-BERT)... Có thể mất 2-3 phút.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/110 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# 3: HUẤN LUYỆN SVD & ĐO LƯỜNG TOÀN HỆ THỐNG
# ==========================================
print("\n3. Đang huấn luyện SVD và đo sai số hệ thống...")
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(rating_df_clean[['user_id', 'anime_id', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd_model = SVD(n_epochs=30, lr_all=0.005, reg_all=0.1, verbose=False)
svd_model.fit(trainset)

# 3.1: Sai số SVD độc lập
svd_predictions = svd_model.test(testset)
print(f"-> RMSE (Chỉ dùng SVD): {accuracy.rmse(svd_predictions, verbose=False):.4f}")
# ==========================================
# 3.2: TÍNH SAI SỐ HYBRID CHO CẢ 2 MÔ HÌNH (BERT VÀ TF-IDF)
# ==========================================
print("-> Đang tính toán Hybrid RMSE cho cả 2 Engine (Khoảng 10.000 mẫu ngẫu nhiên)...")
sample_testset = testset[:10000]

# Khởi tạo 2 mảng lưu sai số riêng biệt
hybrid_errors_bert = []
hybrid_errors_tfidf = []

# Trích xuất user ratings từ tập Train (để làm lịch sử cho Content-Based)
user_history = {}
for uid, iid, r in trainset.all_ratings():
    raw_user = trainset.to_raw_uid(uid)
    raw_item = trainset.to_raw_iid(iid)
    if raw_user not in user_history: user_history[raw_user] = {}
    user_history[raw_user][raw_item] = r

for uid, iid, actual_rating in sample_testset:
    # Điểm 1: SVD (Chuyên gia đoán điểm - Dùng chung cho cả 2)
    svd_est = svd_model.predict(uid, iid).est

    # Khởi tạo điểm Content-based mặc định (Fallback)
    cb_est_bert = svd_est
    cb_est_tfidf = svd_est

    if uid in user_history and iid in id_to_index:
        target_idx = id_to_index[iid]

        sim_rating_pairs_bert = []
        sim_rating_pairs_tfidf = []

        # Quét qua lịch sử xem của user để lấy độ tương đồng
        for watched_id, watched_rating in user_history[uid].items():
            if watched_id in id_to_index:
                watched_idx = id_to_index[watched_id]

                # Trích xuất độ tương đồng từ 2 ma trận khác nhau
                sim_bert = bert_cosine_sim[target_idx][watched_idx]
                sim_tfidf = tfidf_cosine_sim[target_idx][watched_idx]

                # Điều kiện lọc: BERT ( > 0.1 ), TF-IDF ( > 0.0 )
                if sim_bert > 0.1:
                    sim_rating_pairs_bert.append((sim_bert, watched_rating))
                if sim_tfidf > 0.0:
                    sim_rating_pairs_tfidf.append((sim_tfidf, watched_rating))

        # --- XỬ LÝ ĐIỂM CHO BERT (K=5) ---
        sim_rating_pairs_bert.sort(key=lambda x: x[0], reverse=True)
        top_5_bert = sim_rating_pairs_bert[:5]
        if len(top_5_bert) > 0:
            sim_sum = sum([pair[0] for pair in top_5_bert])
            weighted_rating_sum = sum([pair[0] * pair[1] for pair in top_5_bert])
            cb_est_bert = weighted_rating_sum / sim_sum

        # --- XỬ LÝ ĐIỂM CHO TF-IDF (K=5) ---
        sim_rating_pairs_tfidf.sort(key=lambda x: x[0], reverse=True)
        top_5_tfidf = sim_rating_pairs_tfidf[:5]
        if len(top_5_tfidf) > 0:
            sim_sum = sum([pair[0] for pair in top_5_tfidf])
            weighted_rating_sum = sum([pair[0] * pair[1] for pair in top_5_tfidf])
            cb_est_tfidf = weighted_rating_sum / sim_sum

    # ==========================================
    # LAI GHÉP & TÍNH SAI SỐ BÌNH PHƯƠNG
    # ==========================================
    # 1. Engine Deep Learning (80% SVD + 20% BERT)
    hybrid_est_bert = (svd_est * 0.8) + (cb_est_bert * 0.2)
    hybrid_errors_bert.append((actual_rating - hybrid_est_bert) ** 2)

    # 2. Engine Classic (50% SVD + 50% TF-IDF)
    hybrid_est_tfidf = (svd_est * 0.5) + (cb_est_tfidf * 0.5)
    hybrid_errors_tfidf.append((actual_rating - hybrid_est_tfidf) ** 2)

# Tổng hợp RMSE
rmse_bert = math.sqrt(sum(hybrid_errors_bert) / len(hybrid_errors_bert))
rmse_tfidf = math.sqrt(sum(hybrid_errors_tfidf) / len(hybrid_errors_tfidf))

print("\n--- BÁO CÁO SAI SỐ TOÀN HỆ THỐNG ---")
print(f"-> 1. RMSE Độc lập (Chỉ SVD)                : {accuracy.rmse(svd_predictions, verbose=False):.4f}")
print(f"-> 2. RMSE Hybrid (Classic: TF-IDF + SVD)  : {rmse_tfidf:.4f}")
print(f"-> 3. RMSE Hybrid (Deep Learning: BERT+SVD): {rmse_bert:.4f}")
print("-----------------------------------------")


3. Đang huấn luyện SVD và đo sai số hệ thống...
-> RMSE (Chỉ dùng SVD): 1.2097
-> Đang tính toán Hybrid RMSE cho cả 2 Engine (Khoảng 10.000 mẫu ngẫu nhiên)...

--- BÁO CÁO SAI SỐ TOÀN HỆ THỐNG ---
-> 1. RMSE Độc lập (Chỉ SVD)                : 1.2097
-> 2. RMSE Hybrid (Classic: TF-IDF + SVD)  : 1.2605
-> 3. RMSE Hybrid (Deep Learning: BERT+SVD): 1.2273
-----------------------------------------


In [ ]:
# # 4: HÀM HYBRID GỢI Ý (MULTI-INPUT BY ID)
# def multi_input_hybrid_recommendation_by_id(user_ratings_dict):
#     candidate_scores = {}
#     watched_anime_ids = list(user_ratings_dict.keys())

#     for anime_id, rating in user_ratings_dict.items():
#         if anime_id in id_to_index:
#             idx = id_to_index[anime_id]
#             sim_scores = list(enumerate(cosine_sim[idx]))
#             sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:21]

#             for i, sim in sim_scores:
#                 candidate_id = anime_df.iloc[i]['anime_id']
#                 if candidate_id in watched_anime_ids:
#                     continue

#                 score = sim * rating
#                 if candidate_id in candidate_scores:
#                     candidate_scores[candidate_id] += score
#                 else:
#                     candidate_scores[candidate_id] = score

#     top_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:10]
#     result_indices = [id_to_index[cid] for cid, score in top_candidates]
#     result_df = anime_df.iloc[result_indices][['anime_id', 'name', 'genre']].copy()
#     result_df['hybrid_score'] = [score for cid, score in top_candidates]

#     return result_df

In [ ]:
# # 5: CHẠY THỬ NGHIỆM (DEMO)

# danh_sach_myanimelist_id = {
#     19815: 10.0,  # No Game No Life
#     30831: 10.0,  # KonoSuba
#     1535: 606,    # Death Note
#     38691: 9.0    # Dr. Stone: Stone Wars
# }

# print("\n=> KẾT QUẢ GỢI Ý CHO DANH SÁCH BẰNG ID:")
# print(multi_input_hybrid_recommendation_by_id(danh_sach_myanimelist_id))


=> KẾT QUẢ GỢI Ý CHO DANH SÁCH BẰNG ID:
      anime_id                                               name  \
896       2994                                Death Note: Rewrite   
2775     32827                                   B: The Beginning   
2359     24781                       Imawa no Kuni no Alice (OVA)   
3408     40046                                         Id:Invaded   
442        789                               Shinigami no Ballad.   
968       3588                                         Soul Eater   
830       2449  Koukaku Kidoutai: Stand Alone Complex - The La...   
1088      4879                                    Mouryou no Hako   
357        553                                    Yami no Matsuei   
221        323                                    Mousou Dairinin   

                                                  genre  hybrid_score  
896   Mystery, Police, Psychological, Supernatural, ...    216.848664  
2775  Action, Mystery, Police, Psychological, Supern...

In [ ]:
# ==========================================
# 4: XUẤT CÁC FILE ĐỂ ĐEM XUỐNG BACKEND
# ==========================================
print("\n4. Đang xuất file .pkl và tải về máy...")
import pickle
from google.colab import files # Bắt buộc phải import cái này để tải về

# 1. Lưu các model ra file
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)

with open('tfidf_cosine_sim.pkl', 'wb') as f: # File của mô hình cũ
    pickle.dump(tfidf_cosine_sim, f)

with open('bert_cosine_sim.pkl', 'wb') as f:  # File của Deep Learning (Mới)
    pickle.dump(bert_cosine_sim, f)

# 2. Đóng gói Dataframe (LƯU Ý: Phải kẹp thêm cột quality_weight vào)
anime_artifacts = {
    'dataframe': anime_df[['anime_id', 'name', 'genre', 'quality_weight']],
    'id_to_index': id_to_index
}
with open('anime_artifacts.pkl', 'wb') as f:
    pickle.dump(anime_artifacts, f)

# 3. Kích hoạt trình duyệt tải 4 file xuống
print("Đang kích hoạt tải về...")
files.download('svd_model.pkl')
files.download('tfidf_cosine_sim.pkl')
files.download('bert_cosine_sim.pkl')
files.download('anime_artifacts.pkl')



4. Đang xuất file .pkl và tải về máy...
Đang kích hoạt tải về...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>